# SPATIAL INTELLIGENCE - PART 1

In [1]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries

In [2]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

c:\Users\tuemi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the TopologicPy Version

In [3]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.18) is OLDER than the latest version (0.9.20) from PyPI. Please consider upgrading to the latest version.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [4]:
renderer = "vscode"

## 4. Import the OBJ file

In [16]:
objects = Topology.ByOBJPath(r"C:\Users\tuemi\Downloads\AIA_GraphML_ChloeGeometry2.obj")
# print("Objects is a list")
print(objects)


## 6. Show the geometry

In [17]:
Topology.Show(objects,
              faceColor=[210,210,250],
            #   faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              faceOpacity=0.3,
              showVertices=False,
              backgroundColor="white",
              width=800,
              height=600,
              renderer = renderer)

In [18]:
Topology.Show(objects,
              edgeColor=[100,100,200],   # visible color, not white
              edgeWidth=2,
              showFaces=False,           # skip faces entirely
              showVertices=False,
              backgroundColor="white",
              width=800,
              height=600,
              renderer=renderer)

In [19]:
print("Creating CellComplex from faces...")
faces = Topology.Faces(objects[0])
print(f"Number of faces: {len(faces)}")

cellComplex = CellComplex.ByFaces(faces, tolerance=0.0001)

cells = Topology.Cells(cellComplex)
print(f"Number of cells (rooms) created: {len(cells)}")

Creating CellComplex from faces...
Number of faces: 1680
Number of cells (rooms) created: 320


In [20]:
shell = CellComplex.ExternalBoundary(cellComplex)

Topology.Show(shell,
              faceColor=[100, 150, 220],
              faceOpacity=0.3,
              edgeColor=[50, 50, 150],
              edgeWidth=2,
              showVertices=False,
              backgroundColor="white",
              width=800,
              height=600,
              renderer=renderer)

In [21]:
# The imported geometry is a Cluster of faces, not cells
# We need to create a CellComplex which will automatically detect and create cells from the faces

print("Creating CellComplex from faces...")
faces = Topology.Faces(objects[0])
print(f"Number of faces: {len(faces)}")

# Create a CellComplex - this will automatically detect closed volumes (rooms) from the faces
cellComplex = CellComplex.ByFaces(faces, tolerance=0.0001)

# Get all cells (rooms)
cells = Topology.Cells(cellComplex)
print(f"Number of cells (rooms) created: {len(cells)}")

# Find the corridor - it's the cell with the most adjacent cells (highest connectivity)
print("\nAnalyzing cells to find the corridor...")

# Calculate adjacencies for each cell
cell_info = []
for i, cell in enumerate(cells):
    # Count adjacent cells (cells that share a face with this cell)
    adjacent_count = 0
    for j, other_cell in enumerate(cells):
        if i != j:
            # Check if they share faces
            shared_faces = Topology.SharedFaces(cell, other_cell)
            if len(shared_faces) > 0:
                adjacent_count += 1
    
    volume = Cell.Volume(cell)
    cell_info.append((i, volume, adjacent_count, cell))

# Sort by number of adjacent cells (corridor should have most connections)
cell_info.sort(key=lambda x: x[2], reverse=True)

# The corridor is the cell with the most adjacent cells
corridor_idx, corridor_vol, corridor_adjacencies, corridor = cell_info[0]
print(f"Corridor identified: Cell {corridor_idx}")
print(f"  - Volume: {corridor_vol:.2f}")
print(f"  - Adjacent to {corridor_adjacencies} cells")

# Create custom graph with star topology
# ONLY include corridor and rooms directly adjacent to it
corridor_center = Topology.CenterOfMass(corridor)
graph_vertices = [corridor_center]
graph_edges = []

# Only add rooms that are directly adjacent to the corridor
for i, vol, adj, cell in cell_info:
    if i != corridor_idx:  # Skip the corridor itself
        # Check if this room is adjacent to the corridor
        shared_faces = Topology.SharedFaces(cell, corridor)
        if len(shared_faces) > 0:
            # This room connects to the corridor - add it to the graph
            room_center = Topology.CenterOfMass(cell)
            graph_vertices.append(room_center)
            
            # Create edge from room to corridor
            edge = Edge.ByVertices([room_center, corridor_center])
            graph_edges.append(edge)

# Build the star graph
g1 = Graph.ByVerticesEdges(graph_vertices, graph_edges)

print(f"\nStar graph created:")
print(f"  - Vertices: {len(graph_vertices)} (1 corridor + {len(graph_edges)} connected rooms)")
print(f"  - Edges: {len(graph_edges)} (all rooms connect to corridor)")

Creating CellComplex from faces...
Number of faces: 1680
Number of cells (rooms) created: 320

Analyzing cells to find the corridor...
Corridor identified: Cell 18
  - Volume: 8820.00
  - Adjacent to 11 cells

Star graph created:
  - Vertices: 12 (1 corridor + 11 connected rooms)
  - Edges: 11 (all rooms connect to corridor)


In [31]:
# Show graph with building geometry (faces invisible, light edges)
shell = CellComplex.ExternalBoundary(cellComplex)

Topology.Show(shell,
              vertexSize=15,
              vertexColor="red",
              edgeWidth=5,
              edgeColor="blue",
              faceOpacity=0.5,  # Completely invisible faces
              showEdges=True,  # Show building edges as wireframe
              width=800,
              height=600,
              backgroundColor="white",
              renderer=renderer)

In [28]:
# Alternative: Show ONLY the adjacency graph without building geometry
Graph.Show(g_full,
           vertexSize=20,
           vertexColor="red",
           edgeWidth=5,
           edgeColor="blue",
           backgroundColor="white",
           width=800,
           height=600,
           renderer=renderer)

In [37]:
# Create graph for internal room connections only (exclude external boundary faces)
print("Creating internal adjacency graph...")

# Get external boundary faces to exclude them
shell = CellComplex.ExternalBoundary(cellComplex)
external_faces = Topology.Faces(shell)
print(f"External boundary faces: {len(external_faces)}")

# Create vertices at the center of each room
graph_vertices_all = []
cell_to_vertex_map = {}

for i, cell in enumerate(cells):
    center = Topology.CenterOfMass(cell)
    graph_vertices_all.append(center)
    cell_to_vertex_map[i] = center

print(f"Created {len(graph_vertices_all)} vertices at room centers")

# Create edges only between rooms that share INTERNAL faces
graph_edges_all = []
for i, cell1 in enumerate(cells):
    for j, cell2 in enumerate(cells):
        if i < j:
            shared_faces = Topology.SharedFaces(cell1, cell2)
            if len(shared_faces) > 0:
                # Check if ANY shared face is internal (not on external boundary)
                has_internal_face = False
                for sf in shared_faces:
                    is_external = False
                    for ef in external_faces:
                        if Topology.IsSame(sf, ef):
                            is_external = True
                            break
                    if not is_external:
                        has_internal_face = True
                        break
                if has_internal_face:
                    edge = Edge.ByVertices([cell_to_vertex_map[i], cell_to_vertex_map[j]])
                    graph_edges_all.append(edge)

print(f"Created {len(graph_edges_all)} internal edges")

# Build the internal-only graph
g_full = Graph.ByVerticesEdges(graph_vertices_all, graph_edges_all)

print(f"\nInternal adjacency graph:")
print(f"  - Vertices (rooms): {len(graph_vertices_all)}")
print(f"  - Edges (internal connections): {len(graph_edges_all)}")

Creating internal adjacency graph...
External boundary faces: 392
Created 320 vertices at room centers
Created 924 internal edges

Internal adjacency graph:
  - Vertices (rooms): 320
  - Edges (internal connections): 924


## 7. Create Complete Graph for All Floors
Create a graph that includes all rooms across all floors, with edges representing adjacencies.

In [42]:
from topologicpy.Plotly import Plotly

# Layer 1: Building geometry — black wireframe, nearly transparent faces
building_data = Plotly.DataByTopology(
    topology=cellComplex,
    showVertices=False,
    showEdges=True,
    edgeColor="black",
    edgeWidth=1,
    showFaces=True,
    faceOpacity=0.05,
    faceColor="white"
)

# Layer 2: Graph — extract the underlying topology from the graph
graph_topology = Graph.Topology(g_full)
graph_data = Plotly.DataByTopology(
    topology=graph_topology,
    showVertices=True,
    vertexSize=10,
    vertexColor="red",
    showEdges=True,
    edgeWidth=3,
    edgeColor="blue",
    showFaces=False
)

# Combine both layers into one figure
data = building_data + (graph_data or [])
fig = Plotly.FigureByData(data, width=800, height=800, backgroundColor="white")
Plotly.Show(fig, renderer=renderer)